# 🎵 ICA Audio Unmixing: The Cocktail Party Problem

## 📚 Overview

In this notebook, you'll use **Independent Component Analysis (ICA)** to solve the cocktail party problem!

**The Challenge:**
You have 3 mixed audio recordings (x1, x2, x3). Each recording contains a combination of 3 different music tracks mixed together. Your goal is to separate these mixtures back into the original independent sources.

**What you'll do:**
1. ✅ Listen to the mixed audio recordings
2. ✅ Visualize the mixed waveforms
3. ✅ Apply FastICA to unmix the signals
4. ✅ Listen to the recovered sources
5. ✅ Analyze the separation quality

**The Cocktail Party Problem:**
Imagine 3 speakers playing different music simultaneously, recorded by 3 microphones at different positions. Each microphone captures a different linear combination of the 3 sources. Can you recover the original sources?

**Answer:** Yes! Using ICA! 🎉

---

## Part I: Load and Explore the Mixed Audio

Let's start by loading the mixed audio recordings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import FastICA
from IPython.display import Audio, display
import os
import librosa
import soundfile as sf
# Try to import librosa (supports mp3), fallback to soundfile
try:

    USE_LIBROSA = True
    print("✓ Using librosa (supports MP3)")
except ImportError:

    USE_LIBROSA = False
    print("✓ Using soundfile (WAV only)")

# --- Load the mixed audio files ---
mixture_files = [
    "sound/mixture_1.wav",
    "sound/mixture_2.wav",
    "sound/mixture_3.wav"
]

print(f"\n🔊 Loading {len(mixture_files)} mixed audio files...\n")

mixtures = []
sample_rate = None

for i, file_path in enumerate(mixture_files):
    if USE_LIBROSA:
        audio, sr = librosa.load(file_path, sr=None, mono=True)
    else:
        audio, sr = sf.read(file_path)
        if audio.ndim > 1:  # stereo -> mono
            audio = audio[:, 0]
    
    mixtures.append(audio)
    sample_rate = sr
    print(f"Mixture {i+1}: {os.path.basename(file_path)}")
    print(f"  - Duration: {len(audio)/sr:.2f}s")
    print(f"  - Sample rate: {sr} Hz")
    print(f"  - Samples: {len(audio)}\n")

# Align lengths (trim to shortest)
min_len = min(len(m) for m in mixtures)
mixtures = [m[:min_len] for m in mixtures]

# Stack into matrix: shape (n_samples, n_mixtures)
X_mixed = np.column_stack(mixtures)

print(f"✓ Mixture matrix shape: {X_mixed.shape}")
print(f"✓ Duration: {min_len/sample_rate:.2f} seconds")

### 🎧 Listen to the Mixed Audio

Each mixture contains all three sources combined with different weights. Can you hear multiple music tracks overlapping?

In [ ]:
print("🔊 Mixed Audio Recordings:\n")
for i in range(X_mixed.shape[1]):
    print(f"Mixture {i+1}")
    display(Audio(X_mixed[:, i], rate=sample_rate))
    print()

print("💡 Notice: Each mixture sounds like a complex overlay of multiple tracks!")

### 📊 Visualize the Mixed Waveforms

Let's look at the time-domain waveforms of the mixed signals:

In [ ]:
# Plot first 3 seconds for clarity
plot_duration = 3  # seconds
plot_samples = int(plot_duration * sample_rate)

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

for i in range(3):
    axes[i].plot(
        np.arange(plot_samples) / sample_rate, 
        X_mixed[:plot_samples, i], 
        color='purple',
        linewidth=0.5
    )
    axes[i].set_title(f"Mixture {i+1} (all sources combined)", fontweight='bold')
    axes[i].set_ylabel("Amplitude")
    axes[i].grid(True, alpha=0.3)
    if i == 2:
        axes[i].set_xlabel("Time (s)")

plt.suptitle("Mixed Audio Waveforms", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("📊 Observation:")
print("These waveforms show the mixed signals. Our goal is to separate")
print("these into the individual independent source signals!")

---

## Part II: Apply ICA to Unmix the Signals

Now for the magic! We'll use **FastICA** to recover the independent sources from the mixtures.

### How FastICA Works

**The Mathematical Model:**
$$\mathbf{X} = \mathbf{S} \mathbf{A}^T$$

where:
- $\mathbf{X}$ = observed mixtures (what we have)
- $\mathbf{S}$ = unknown independent sources (what we want)
- $\mathbf{A}$ = unknown mixing matrix

**ICA Goal:** Given only $\mathbf{X}$, find $\mathbf{S}$!

### The Algorithm Steps:

1. **Preprocessing:**
   - **Centering:** Remove mean → $\mathbf{X}_{\text{centered}} = \mathbf{X} - \mathbb{E}[\mathbf{X}]$
   - **Whitening:** Decorrelate and normalize → spherical data distribution

2. **Iterative Optimization:**
   - Find directions of **maximum non-Gaussianity**
   - Use fixed-point iteration (Newton's method)
   - Each direction corresponds to one independent component

3. **Output:**
   - Unmixing matrix $\mathbf{W}$
   - Recovered sources: $\mathbf{S}_{\text{est}} = \mathbf{X} \mathbf{W}^T$

**Key Principle:** Independent sources are more non-Gaussian than their mixtures (Central Limit Theorem)!

---

Let's apply FastICA:

In [ ]:
# --- Apply FastICA ---
print("🔍 Applying FastICA to unmix the signals...\n")

ica = FastICA(
    n_components=3,           # Number of sources to recover
    whiten='unit-variance',   # Whitening method
    max_iter=5000,            # Maximum iterations
    tol=1e-4,                 # Convergence tolerance
    random_state=42           # For reproducibility
)

S_recovered = ica.fit_transform(X_mixed)

# Normalize recovered signals
S_recovered = S_recovered / np.max(np.abs(S_recovered), axis=0) * 0.9

print("✓ ICA completed successfully!")
print(f"✓ Recovered sources shape: {S_recovered.shape}")
print(f"✓ Estimated mixing matrix shape: {ica.mixing_.shape}")
print(f"✓ Converged in {ica.n_iter_} iterations")
print("\n" + "="*60)
print("The algorithm has separated the mixtures into 3 independent sources!")
print("="*60)

---

## Part III: Explore the Recovered Sources

Let's visualize and listen to the recovered independent sources!

### 📊 Visualize Recovered Waveforms

In [ ]:
# Plot recovered sources (first 3 seconds)
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

colors = ['darkred', 'darkblue', 'darkgreen']
for i in range(3):
    axes[i].plot(
        np.arange(plot_samples) / sample_rate, 
        S_recovered[:plot_samples, i], 
        color=colors[i],
        linewidth=0.5
    )
    axes[i].set_title(f"Recovered Source {i+1}", fontweight='bold')
    axes[i].set_ylabel("Amplitude")
    axes[i].grid(True, alpha=0.3)
    if i == 2:
        axes[i].set_xlabel("Time (s)")

plt.suptitle("ICA Recovered Sources", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("📊 Observation:")
print("Each waveform now shows a clean, independent source!")
print("Compare these to the mixed signals above - much cleaner!")

### 🎧 Listen to the Recovered Sources

**The moment of truth:** Do the recovered signals sound like clean, separated music tracks?

**Important Notes:**
- ICA has three fundamental ambiguities:
  1. **Order:** Recovered sources may be in any order
  2. **Sign:** Signals may be inverted (multiplied by -1)
  3. **Scale:** Amplitudes may differ

These ambiguities don't affect the separation quality - the sources are still perfectly separated!

In [ ]:
print("🎶 Recovered Independent Sources:\n")
for i in range(S_recovered.shape[1]):
    print(f"Recovered Source {i+1}")
    display(Audio(S_recovered[:, i], rate=sample_rate))
    print()

print("\n" + "="*60)
print("💡 LISTENING TEST:")
print("="*60)
print("Each recovered source should sound like a clean, individual music track!")
print("Compare these to the mixed signals from Part I.")
print("Notice how ICA successfully separated the overlapping audio!")
print("="*60)

### 💾 Save Recovered Sources

Let's save the recovered sources to disk:

In [ ]:
# Save recovered sources
recovered_dir = "sound/"
os.makedirs(recovered_dir, exist_ok=True)

for i in range(S_recovered.shape[1]):
    filename = f"{recovered_dir}/recovered_source_{i+1}.wav"
    
    if USE_LIBROSA:
        import soundfile as sf
        sf.write(filename, S_recovered[:, i], sample_rate)
    else:
        sf.write(filename, S_recovered[:, i], sample_rate)
    
    print(f"✓ Saved: {filename}")

print(f"\n✓ All recovered sources saved to: {recovered_dir}/")

---

## Part IV: Analyze the Results

Let's examine the ICA output more closely:

### Estimated Mixing Matrix

ICA also estimates the mixing matrix that was used to create the mixtures:

In [ ]:
print("🔍 Estimated Mixing Matrix (from ICA):\n")
print(ica.mixing_)
print("\nInterpretation:")
print("Each row shows how one mixture was formed from the three sources.")
print("For example:")
print(f"Mixture 1 ≈ {ica.mixing_[0,0]:.2f}*Src1 + {ica.mixing_[0,1]:.2f}*Src2 + {ica.mixing_[0,2]:.2f}*Src3")
print(f"Mixture 2 ≈ {ica.mixing_[1,0]:.2f}*Src1 + {ica.mixing_[1,1]:.2f}*Src2 + {ica.mixing_[1,2]:.2f}*Src3")
print(f"Mixture 3 ≈ {ica.mixing_[2,0]:.2f}*Src1 + {ica.mixing_[2,1]:.2f}*Src2 + {ica.mixing_[2,2]:.2f}*Src3")

print("\n💡 This shows how the sources were combined to create the mixtures!")

### Independence Check

Let's verify that the recovered sources are statistically independent by checking their correlation:

In [ ]:
# Compute correlation matrix between recovered sources
print("📊 Correlation Matrix of Recovered Sources:\n")

corr_recovered = np.corrcoef(S_recovered.T)

print("        Src1   Src2   Src3")
for i in range(3):
    print(f"Src{i+1}: ", end="")
    for j in range(3):
        print(f"{corr_recovered[i,j]:6.3f} ", end="")
    print()

print("\n💡 Interpretation:")
print("Diagonal elements = 1.0 (each source correlates perfectly with itself)")
print("Off-diagonal elements ≈ 0 (sources are uncorrelated/independent)")
print("\nIf off-diagonal values are close to 0, ICA successfully found")
print("independent components!")

# Check maximum off-diagonal correlation
off_diag = corr_recovered[~np.eye(3, dtype=bool)]
max_corr = np.max(np.abs(off_diag))
print(f"\n🎯 Maximum correlation between different sources: {max_corr:.3f}")
if max_corr < 0.1:
    print("✓ Excellent separation! Sources are highly independent.")
elif max_corr < 0.3:
    print("✓ Good separation! Sources are mostly independent.")
else:
    print("⚠ Moderate separation. Some correlation remains.")

---

## 📊 Summary and Key Takeaways

### What We Accomplished

1. ✅ **Loaded** 3 mixed audio recordings containing overlapped sources
2. ✅ **Visualized** the complex mixed waveforms  
3. ✅ **Applied FastICA** to unmix the signals
4. ✅ **Recovered** 3 independent source signals
5. ✅ **Verified** separation quality (visualization + listening test)
6. ✅ **Analyzed** the estimated mixing matrix
7. ✅ **Saved** recovered sources to disk

### 🎯 Key ICA Principles

| Concept | Explanation |
|---------|-------------|
| **Blind Source Separation** | Recover sources without knowing the mixing process |
| **Linear Mixing Model** | $\mathbf{X} = \mathbf{S} \mathbf{A}^T$ |
| **Statistical Independence** | Sources are statistically independent |
| **Non-Gaussianity Criterion** | ICA finds directions of maximum non-Gaussianity |
| **Whitening** | Preprocessing to decorrelate mixtures |
| **Fixed-Point Iteration** | FastICA's optimization algorithm |
| **Central Limit Theorem** | Mixtures are more Gaussian than sources |

### 🔬 The Three ICA Ambiguities

**Remember:** ICA cannot determine:
1. **Order (Permutation):** Which recovered source corresponds to which original
2. **Sign (Polarity):** Whether to multiply by +1 or -1
3. **Scale (Amplitude):** The absolute scaling factor

**Important:** These are fundamental limitations, not bugs! In most applications (including audio separation), they don't matter because we only care about **separating** the sources, not identifying their exact scale or order.

### 📁 Output Files

**Input (mixtures):**
- `sound/mixed_output/mixture_1.wav`
- `sound/mixed_output/mixture_2.wav`
- `sound/mixed_output/mixture_3.wav`

**Output (recovered sources):**
- `sound/recovered_output/recovered_source_1.wav`
- `sound/recovered_output/recovered_source_2.wav`
- `sound/recovered_output/recovered_source_3.wav`

### 🚀 Real-World Applications

This same ICA technique is used in:

- **Audio Processing:** 
  - Separating overlapping speakers (cocktail party problem)
  - Removing background noise from recordings
  - Music source separation (vocals, drums, etc.)

- **Medical Imaging:**
  - Removing eye-blink artifacts from EEG signals
  - Separating brain activity patterns in fMRI
  - Heart signal analysis in ECG

- **Finance:**
  - Identifying independent market factors
  - Risk analysis and portfolio optimization

- **Telecommunications:**
  - Blind signal separation in wireless communications
  - Removing interference from signals

- **Image Processing:**
  - Separating overlapped textures
  - Hyperspectral image analysis

### 🎓 Key Learning Points

1. **ICA exploits statistical independence** to separate mixed signals
2. **Non-Gaussianity is the key** - independent sources are more non-Gaussian than mixtures
3. **Preprocessing (centering + whitening)** is crucial for ICA success
4. **The three ambiguities** (order, sign, scale) are fundamental but usually don't matter
5. **FastICA is efficient** - typically converges in tens to hundreds of iterations

### 🎉 Congratulations!

You've successfully solved the cocktail party problem using ICA!

**Key insight:** By assuming statistical independence and maximizing non-Gaussianity, ICA can separate complex mixtures back into their original independent sources - without any prior knowledge of the mixing process!

This is the power of **Blind Source Separation**! 🎵✨